# SRCNN 255×255: offline evaluation and KV260 deployment

Hardware: frozen 200 MHz overlay. Status: board execution pending. This notebook keeps software/reference results separate from physical FPGA measurements. Input must already be bicubic-interpolated and normalized; no implicit resizing or output clamp.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
ROOT = Path.cwd()
if not (ROOT / 'tools').is_dir():
    ROOT = ROOT.parent
assert (ROOT / 'tools' / 'prepare_srcnn_deployment.py').is_file(), 'Open from repository root or pynq directory'
sys.path.insert(0, str(ROOT / 'tools'))
sys.path.insert(0, str(ROOT / 'pynq'))
BUNDLE = ROOT / 'data' / 'butterfly_deploy'
FLOAT_DATA = ROOT / 'data' / 'butterfly_float'
BITFILE = Path('/home/xilinx/srcnn_axis_255x255.bit')
RESULT_DIR = ROOT / 'board-results' / 'notebook-run-001'
RUN_PREPARE = False
RUN_ON_BOARD = False
CROP = 0  # Explicit policy; confirm course border-crop convention
ITERATIONS = 3


## 1. Verify and import formal course assets

On the tool machine first run `python tools/import_course_assets.py --root PATH_TO_GOLDEN_ROOT --out data/butterfly_float`. It checks the ten recorded SHA-256 hashes. Butterfly is one test case; it does not represent all Set5 images.


In [ ]:
from prepare_srcnn_deployment import prepare
if RUN_PREPARE:
    offline = prepare(FLOAT_DATA / 'input_float.npy', FLOAT_DATA / 'model_float.npy', BUNDLE,
                      FLOAT_DATA / 'course_golden.npy', FLOAT_DATA / 'ground_truth.npy', CROP)
else:
    print('Preparation disabled. Enable only after formal assets are imported; output directory must be new.')


## 2. Inspect offline precision and image quality

Q24.8 raw words use ties-to-even rounding and saturation. Separate fixed-versus-Golden error from reconstruction quality versus HR. Peak=1; crop is recorded. No clipping/rounding is used in MSE/PSNR. Perfect matches have `psnr_db=null` and `perfect_match=true` (+infinity).


In [ ]:
from srcnn_axis_host import read_bundle
manifest, arrays = None, None
if BUNDLE.is_dir():
    manifest, arrays = read_bundle(BUNDLE)
    offline = json.loads((BUNDLE / 'offline_metrics.json').read_text())
    print(json.dumps(offline, indent=2))
    print('Bundle status:', manifest['status'])
else:
    print('Formal bundle unavailable; no official quality values reported.')


## 3. Physical KV260 validation

Set `RUN_ON_BOARD=True` only on the KV260 PYNQ image with matching bit/HWH and a verified bundle. The runner checks current overlay hashes, DMA capacity, 200 MHz PL0, model pointer, completion/error/timeout and every output word. This is the only section that can establish board PASS. If an exception reports PL reset failure, keep the runner/session alive and reset the board before releasing retained buffers.


In [ ]:
from srcnn_axis_host import Runner
runner, board_results = None, None
if RUN_ON_BOARD:
    assert arrays is not None, 'Prepare and verify the formal bundle first'
    RESULT_DIR.mkdir(parents=True, exist_ok=False)
    board_results = {'status': 'RUNNING', 'runs': [], 'bundle': manifest}
    try:
        runner = Runner(BITFILE, timeout=120)
        for index in range(ITERATIONS):
            result, output = runner.run(arrays)
            board_results['runs'].append(result)
            np.save(RESULT_DIR / f'output_{index}.npy', output, allow_pickle=False)
        board_results['status'] = 'PASS'
    except Exception as error:
        board_results.update(status='FAIL', error=str(error))
        raise
    finally:
        (RESULT_DIR / 'board_results.json').write_text(json.dumps(board_results, indent=2))
else:
    print('BOARD NOT RUN. No measured FPGA FPS.')


## 4. Report timing boundaries and repeatability

Transfer-window timing includes model loading and DMA. Host timing includes allocation/cache/verification and excludes file loading, bicubic preprocessing and display. Neither is pure kernel-only FPS. Report the actual PL clock and all repetitions.


In [ ]:
if board_results and board_results['status'] == 'PASS':
    seconds = np.array([r['transfer_start_to_done_s'] for r in board_results['runs']])
    print('Frames:', len(seconds), 'exact comparisons: PASS')
    print('Transfer-window median seconds:', float(np.median(seconds)))
    print('Transfer-window median FPS:', float(1 / np.median(seconds)))
    print('PL0 MHz:', board_results['runs'][0]['identity']['pl0_mhz'])
else:
    print('Physical performance measurements pending.')


## 5. Full Set5 aggregation

Prepare all five official cases with identical preprocessing/peak/crop. Create a JSON mapping `baby`, `bird`, `butterfly`, `head`, `woman` to their bundle directories, then run `python tools/summarize_set5.py --cases data/set5_cases.json --out data/set5_offline_results.json`. All cases need both course Golden and HR data. Mean PSNR is the mean of per-image dB values. Offline reference inference is distinct from FPGA image-quality results.
